# Debugging => Tracebacks, assert, breakpoint & pdb

Debugging means finding **why** the code does not do what you expect. Start with the traceback, then narrow the problem.

| Tool | Purpose |
|---|---|
| Traceback | Shows the call chain that led to an error |
| `print()` / `f"{x=}"` | Quick look at values |
| `repr(x)` | Shows the exact value, including quotes and escapes |
| `assert condition, message` | Stop if an assumption is false |
| `traceback.format_exc()` | Get the traceback as text inside `except` |
| `breakpoint()` | Pause and open the debugger |
| `pdb` | The built-in interactive debugger |
| `python -m pdb script.py` | Start a script under the debugger |
| `%debug` | IPython/Jupyter: debug the last error (post-mortem) |

---

## Reading a Traceback

```text
Traceback (most recent call last):
  File "app.py", line 12, in <module>
    result = process(data)
  File "app.py", line 8, in process
    return total / count
ZeroDivisionError: division by zero
```

* Read from the **bottom**: the last line is the exception type and message.
* The frame just above it is where the error **happened**.
* Frames higher up show **who called** it.
* "most recent call last" means the newest call is at the bottom.

---

## Printing Values

```python
print(f"{total=}, {count=}")
print(repr(text))            # shows quotes, spaces and escapes such as \n
```

`repr()` reveals problems that `print()` hides: trailing spaces, `"5"` vs `5`, `None` vs `"None"`.

---

## `assert`

```python
assert count > 0, "count must be positive"
```

* Raises `AssertionError` when the condition is false.
* Use it to check **your own assumptions** while developing.
* Assertions are **removed** when Python runs with `-O`. Never use `assert` to validate user input. Raise a real exception instead.

---

## `breakpoint()` and `pdb`

`breakpoint()` pauses the program and opens the debugger at that line.

| Command | Meaning |
|---|---|
| `n` (next) | Run the current line, stay in this function |
| `s` (step) | Step into a function call |
| `c` (continue) | Run until the next breakpoint |
| `l` (list) | Show the code around the current line |
| `p expr` / `pp expr` | Print / pretty-print an expression |
| `w` (where) | Show the call stack |
| `u` / `d` | Move up / down the stack |
| `b line` | Set a breakpoint |
| `r` (return) | Run until the current function returns |
| `q` (quit) | Leave the debugger |
| `h` | Help |

### Important

* Set the environment variable `PYTHONBREAKPOINT=0` to disable every `breakpoint()` call.
* `pdb` needs an interactive terminal. It cannot run in an unattended script.

---

## A Debugging Routine

1. **Reproduce** the problem with the smallest input.
2. **Read** the traceback from the bottom.
3. **Form a guess**, then check it with `print`, `assert` or `pdb`.
4. **Change one thing** at a time.
5. **Add a test** so the bug cannot return (see Unit Testing).

## Source

https://docs.python.org/3/library/pdb.html

https://docs.python.org/3/library/traceback.html

https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement

In [ ]:
import traceback

def average(values):
    return sum(values) / len(values)

def report(values):
    return f"average={average(values)}"

# Capture a traceback as text and read it from the bottom
try:
    report([])
except ZeroDivisionError:
    text = traceback.format_exc()

lines = text.splitlines()
print(lines[0])                    # Traceback (most recent call last):
print(lines[-1])                   # the exception type and message

# The frames, from the outermost call to where the error happened
try:
    report([])
except ZeroDivisionError as error:
    frames = traceback.extract_tb(error.__traceback__)
print([frame.name for frame in frames])

# print(f"{x=}") and repr() reveal what print() hides
total, count = 10, 0
print(f"{total=}, {count=}")
value = "5 "
print(value, "|", repr(value), "|", value == "5")

# assert checks your own assumptions
def safe_average(values):
    assert len(values) > 0, "values must not be empty"
    return sum(values) / len(values)

try:
    safe_average([])
except AssertionError as error:
    print("AssertionError:", error)

print(safe_average([2, 4, 6]))

# assert can be removed with `python -O`, so validate real input with exceptions
def set_age(age):
    if age < 0:
        raise ValueError("age must not be negative")
    return age

try:
    set_age(-1)
except ValueError as error:
    print("ValueError:", error)

# breakpoint() is not called here: it would wait for keyboard input.
# In a script:   breakpoint()         (opens pdb at that line)
# Disable all:   PYTHONBREAKPOINT=0 python script.py
# In Jupyter:    run %debug after an error to inspect it
import builtins
print(callable(builtins.breakpoint))